# Notebook 2 — Operational 500 m Pixel Fire-Risk Model

## Goal

This notebook explains the **operational daily 500 m pixel-risk model**.

The model answers:

> For a selected prediction date, which 500 m candidate pixels should be prioritized for fire-risk alerts?

## Output interpretation

The model is used primarily for **ranking**. It produces:

- `raw_risk_score` — best used for sorting pixels and choosing alert budgets
- `calibrated_proxy_probability` — a proxy-calibrated value, not a full national probability
- `alert_tier` — `Critical`, `High`, `Watch`, or `Monitor`

## Important limitation

The current dataset is a historical-fire-cell proxy, not the complete national burnable-land grid. Therefore, the dashboard should use alert budgets such as top 1%, 5%, or 10%, rather than absolute probability thresholds.


## 1. Imports and artifact discovery

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

ROOT = Path.cwd()

def find_artifact(names, folders=("data", "models", ".")) -> Path:
    """Find a project file in either a flat repo layout or data/models folders."""
    if isinstance(names, str):
        names = [names]
    search_dirs = []
    for folder in folders:
        p = ROOT / folder if folder != "." else ROOT
        if p not in search_dirs:
            search_dirs.append(p)
    for folder in search_dirs:
        for name in names:
            candidate = folder / name
            if candidate.exists():
                return candidate
    # Return the first preferred name in the root so error messages are clear.
    return ROOT / names[0]

def require_file(path: Path, purpose: str) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {purpose}: {path}")

def load_json(path: Path) -> dict:
    require_file(path, "JSON artifact")
    return json.loads(path.read_text(encoding="utf-8"))

def display_if_available(obj, max_rows=10):
    """Small helper for notebooks that may run in different environments."""
    try:
        display(obj.head(max_rows) if hasattr(obj, "head") else obj)
    except NameError:
        print(obj.head(max_rows) if hasattr(obj, "head") else obj)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

PIXEL_DATA_PATH = find_artifact([
    "final_trainable_500m_wildfire_dataset_model_aligned.csv",
    "final_trainable_500m_wildfire_dataset.csv",
    "wildfire_pixel_proxy_500m.csv",
])
PIXEL_BUNDLE_PATH = find_artifact("operational_pixel_proxy_model_bundle.joblib")
PIXEL_SCORES_PATH = find_artifact("operational_pixel_scores_all.csv")
PIXEL_METRICS_PATH = find_artifact("operational_pixel_proxy_metrics.json")
PIXEL_IMPORTANCE_PATH = find_artifact("operational_pixel_proxy_feature_importance.csv")

TARGET = "target_fire_1d"
WEIGHT = "case_control_weight"
SPLIT = "data_split"
ALERT_FRACTIONS = {"Critical — top 1%": 0.01, "High — top 5%": 0.05, "Watch — top 10%": 0.10}

print("Pixel model-aligned data:", PIXEL_DATA_PATH)
print("Pixel model bundle:", PIXEL_BUNDLE_PATH)
print("Pixel scores:", PIXEL_SCORES_PATH)
print("Pixel metrics:", PIXEL_METRICS_PATH)


## 2. Load pixel data and define feature policy

The model-aligned dataset already incorporates the earlier data-quality decisions:

- wind direction was dropped
- precipitation gaps were seasonally imputed
- missingness indicators were added
- prior-fire duration was converted into a censored-history feature
- direct label or sampling fields are excluded from predictors


In [ ]:
require_file(PIXEL_DATA_PATH, "model-aligned pixel dataset")
df = pd.read_csv(PIXEL_DATA_PATH, parse_dates=["prediction_date"])

EXCLUDED_FROM_FEATURES = {
    TARGET, WEIGHT, SPLIT, "pixel_id_500m", "prediction_date",
    "selection_probability", "sampling_stratum", "candidate_mask_scope",
    "operational_landscape_ready", "spatial_fold",
}
FEATURE_COLUMNS = [c for c in df.columns if c not in EXCLUDED_FROM_FEATURES]
CATEGORICAL_FEATURES = [c for c in FEATURE_COLUMNS if df[c].dtype == "object"]
NUMERIC_FEATURES = [c for c in FEATURE_COLUMNS if c not in CATEGORICAL_FEATURES]

print(f"Rows: {len(df):,}")
print(f"Features: {len(FEATURE_COLUMNS):,}")
print(f"Categorical: {CATEGORICAL_FEATURES}")
print("Split counts:")
print(df[SPLIT].value_counts(dropna=False))
print("Missing model-input cells:", int(df[FEATURE_COLUMNS].isna().sum().sum()))
display_if_available(df)


## 3. Model and evaluation functions

The preferred model is XGBoost if available, with a histogram gradient boosting fallback.

The training uses `case_control_weight` because the dataset is sampled. Without weights, the model would learn the sampled class balance rather than the intended operational weighting.


In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def make_preprocessor():
    return ColumnTransformer(
        [
            ("cat", make_one_hot_encoder(), CATEGORICAL_FEATURES),
            ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

def make_classifier(train_frame):
    if XGBClassifier is not None:
        y = train_frame[TARGET].astype(int)
        scale_pos_weight = max(1.0, float((y == 0).sum() / max(1, (y == 1).sum())))
        estimator = XGBClassifier(
            n_estimators=350,
            max_depth=4,
            learning_rate=0.045,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            reg_lambda=2.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=4,
            scale_pos_weight=scale_pos_weight,
        )
    else:
        estimator = HistGradientBoostingClassifier(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=16,
            l2_regularization=0.1,
            random_state=42,
        )
    return Pipeline([("preprocess", make_preprocessor()), ("model", estimator)])

def weighted_metrics(y_true, score, sample_weight):
    y = np.asarray(y_true, dtype=int)
    p = np.asarray(score, dtype=float)
    w = np.asarray(sample_weight, dtype=float)
    out = {
        "weighted_prevalence": float(np.average(y, weights=w)),
        "weighted_mean_prediction": float(np.average(p, weights=w)),
        "brier": float(brier_score_loss(y, p, sample_weight=w)),
    }
    if len(np.unique(y)) > 1:
        out["roc_auc"] = float(roc_auc_score(y, p, sample_weight=w))
        out["pr_auc"] = float(average_precision_score(y, p, sample_weight=w))
    return out

def add_alert_columns(frame, alert_fraction):
    out = frame.copy()
    if out.empty:
        out["is_selected_alert"] = False
        return out
    n_alerts = max(1, int(np.ceil(len(out) * alert_fraction)))
    out = out.sort_values("raw_risk_score", ascending=False).reset_index(drop=True)
    out["rank_for_selected_budget"] = np.arange(1, len(out) + 1)
    out["is_selected_alert"] = out["rank_for_selected_budget"] <= n_alerts
    return out

def assign_alert_tiers(scores):
    q99, q95, q90 = scores.quantile(0.99), scores.quantile(0.95), scores.quantile(0.90)
    return pd.Series(
        np.select(
            [scores >= q99, scores >= q95, scores >= q90],
            ["Critical", "High", "Watch"],
            default="Monitor",
        ),
        index=scores.index,
    )


## 4. Train, validate, calibrate, and test

The split design is:

- Train: 2017–2022
- Validation: 2023
- Test: 2024

The validation set is used for calibration. The raw score remains the main alert-ranking signal.


In [ ]:
train_df = df[df[SPLIT] == "train"].copy()
validation_df = df[df[SPLIT] == "validation"].copy()
test_df = df[df[SPLIT] == "test"].copy()

model = make_classifier(train_df)
model.fit(
    train_df[FEATURE_COLUMNS],
    train_df[TARGET].astype(int),
    model__sample_weight=train_df[WEIGHT].to_numpy(dtype=float),
)

validation_raw = model.predict_proba(validation_df[FEATURE_COLUMNS])[:, 1]
test_raw = model.predict_proba(test_df[FEATURE_COLUMNS])[:, 1]

calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
calibrator.fit(validation_raw, validation_df[TARGET].astype(int), sample_weight=validation_df[WEIGHT])

test_calibrated = calibrator.predict(test_raw)

print("Raw test metrics:")
print(json.dumps(weighted_metrics(test_df[TARGET], test_raw, test_df[WEIGHT]), indent=2))
print("\nCalibrated proxy probability test metrics:")
print(json.dumps(weighted_metrics(test_df[TARGET], test_calibrated, test_df[WEIGHT]), indent=2))


## 5. Alert budget evaluation

Alert budgets are more meaningful than fixed probability thresholds for this model.

Examples:

- top 1% = Critical
- top 5% = High
- top 10% = Watch


In [ ]:
test_scored = test_df.copy()
test_scored["raw_risk_score"] = test_raw
test_scored["calibrated_proxy_probability"] = test_calibrated

budget_rows = []
for label, fraction in ALERT_FRACTIONS.items():
    ranked = add_alert_columns(test_scored, fraction)
    alerted = ranked[ranked["is_selected_alert"]]
    total_positive_weight = (ranked[TARGET] * ranked[WEIGHT]).sum()
    tp_weight = (alerted[TARGET] * alerted[WEIGHT]).sum()
    alert_weight = alerted[WEIGHT].sum()
    budget_rows.append({
        "alert_budget": label,
        "alert_fraction": fraction,
        "score_threshold": alerted["raw_risk_score"].min(),
        "weighted_precision": tp_weight / alert_weight if alert_weight else np.nan,
        "weighted_recall": tp_weight / total_positive_weight if total_positive_weight else np.nan,
        "alerted_rows": len(alerted),
    })

alert_budget_df = pd.DataFrame(budget_rows)
display_if_available(alert_budget_df, 20)


## 6. Map preparation logic

The dashboard map uses a dedicated lightweight table so that PyDeck receives only JSON-safe values.

The map colours use a lighter palette:

- pale green = lower risk
- pale yellow = moderate risk
- light coral = higher risk
- larger outlined points = selected alerts


In [ ]:
TUNISIA_LAT_BOUNDS = (30.0, 38.5)
TUNISIA_LON_BOUNDS = (6.0, 12.5)

def interpolate_rgb(start, end, fraction):
    fraction = float(np.clip(fraction, 0.0, 1.0))
    return [int(round(a + (b - a) * fraction)) for a, b in zip(start, end)]

def light_risk_color(normalized_score, selected_alert):
    value = float(np.clip(normalized_score, 0.0, 1.0))
    pale_green = (211, 241, 220)
    pale_yellow = (255, 244, 184)
    pale_coral = (249, 166, 151)
    if value <= 0.5:
        rgb = interpolate_rgb(pale_green, pale_yellow, value / 0.5)
    else:
        rgb = interpolate_rgb(pale_yellow, pale_coral, (value - 0.5) / 0.5)
    return [*rgb, 225 if selected_alert else 185]

def prepare_pixel_map_data(frame):
    required = [
        "pixel_centroid_lat_proxy", "pixel_centroid_lon_proxy",
        "raw_risk_score", "is_selected_alert",
    ]
    if any(c not in frame.columns for c in required):
        return pd.DataFrame()

    optional_defaults = {
        "gouvernorat": "Unknown",
        "pixel_id_500m": "Unknown",
        "alert_tier": "Monitor",
        "calibrated_proxy_probability": np.nan,
    }
    work = frame.copy()
    for column, default in optional_defaults.items():
        if column not in work.columns:
            work[column] = default

    map_df = work[required + list(optional_defaults)].copy()
    map_df["latitude"] = pd.to_numeric(map_df["pixel_centroid_lat_proxy"], errors="coerce")
    map_df["longitude"] = pd.to_numeric(map_df["pixel_centroid_lon_proxy"], errors="coerce")
    map_df["raw_risk_score"] = pd.to_numeric(map_df["raw_risk_score"], errors="coerce")
    map_df["is_selected_alert"] = map_df["is_selected_alert"].fillna(False).astype(bool)

    map_df = map_df.dropna(subset=["latitude", "longitude", "raw_risk_score"])
    map_df = map_df[
        map_df["latitude"].between(*TUNISIA_LAT_BOUNDS)
        & map_df["longitude"].between(*TUNISIA_LON_BOUNDS)
    ].copy()

    if map_df.empty:
        return map_df

    score_min, score_max = map_df["raw_risk_score"].min(), map_df["raw_risk_score"].max()
    if score_max > score_min:
        map_df["normalized_risk"] = (map_df["raw_risk_score"] - score_min) / (score_max - score_min)
    else:
        map_df["normalized_risk"] = 0.5

    map_df["risk_color"] = [
        light_risk_color(score, alert)
        for score, alert in zip(map_df["normalized_risk"], map_df["is_selected_alert"])
    ]
    map_df["point_radius"] = np.where(map_df["is_selected_alert"], 420, 285)
    return map_df.reset_index(drop=True)

# Example for the latest test date if scores are available.
example = test_scored.copy()
example["alert_tier"] = assign_alert_tiers(example["raw_risk_score"])
example = add_alert_columns(example, 0.05)
map_ready = prepare_pixel_map_data(example)
print(f"Map-ready rows: {len(map_ready):,}")
display_if_available(map_ready)


## 7. Loading existing production artifacts

In the deployed app, scoring normally uses the saved model bundle and score table rather than retraining each time.

In [ ]:
if PIXEL_METRICS_PATH.exists():
    pixel_metrics = load_json(PIXEL_METRICS_PATH)
    print("Saved pixel metrics:")
    print(json.dumps(pixel_metrics, indent=2)[:3500])

if PIXEL_SCORES_PATH.exists():
    scores = pd.read_csv(PIXEL_SCORES_PATH, parse_dates=["prediction_date"])
    scores["date"] = scores["prediction_date"].dt.date
    display_if_available(scores)
